In [12]:

# ============================================================
# NOTEBOOK 07 — NDCLD15 DIFFUSION MODEL TRAINING
# Corrected compact DDPM U-Net; unconditional generation
# ============================================================
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, time, zipfile, random, os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image

START = time.time()
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT = Path("/kaggle/working/nd_iris_project")
IMAGE_DIR = PROJECT / "notebook02_preprocessing/images_224"
OUT = PROJECT / "notebook07_diffusion_training"
SAMPLE_DIR = OUT / "generated_samples"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

assert IMAGE_DIR.exists(), f"Missing preprocessed images: {IMAGE_DIR}"
images = sorted(IMAGE_DIR.glob("*.png"))
assert images, "No preprocessed images found."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_count = torch.cuda.device_count()
print(f"Device: {device}; GPUs available: {gpu_count}")
for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Efficient training resolution for T4
IMAGE_SIZE = 64
BATCH_SIZE = 64
EPOCHS = 5
TIMESTEPS = 200
BASE = 32
LR = 2e-4
WORKERS = min(4, os.cpu_count() or 2)

class GrayDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            im = im.convert("L").resize(
                (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS
            )
            arr = np.asarray(im, dtype=np.float32) / 127.5 - 1.0
        return torch.from_numpy(arr[None])

loader = DataLoader(
    GrayDataset(images), batch_size=BATCH_SIZE, shuffle=True,
    num_workers=WORKERS, pin_memory=(device.type == "cuda"),
    drop_last=True
)

def time_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(
        -np.log(10000) * torch.arange(half, device=t.device).float()
        / max(half - 1, 1)
    )
    angles = t.float()[:, None] * freqs[None, :]
    emb = torch.cat([angles.sin(), angles.cos()], dim=1)
    return F.pad(emb, (0, dim - emb.shape[1]))

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, temb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(temb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)

class SmallUNet(nn.Module):
    def __init__(self, base=BASE, time_dim=128):
        super().__init__()
        self.time_dim = time_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim), nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )
        self.input = nn.Conv2d(1, base, 3, padding=1)
        self.enc1 = ResBlock(base, base, time_dim)
        self.down1 = nn.Conv2d(base, base*2, 4, stride=2, padding=1)
        self.enc2 = ResBlock(base*2, base*2, time_dim)
        self.down2 = nn.Conv2d(base*2, base*4, 4, stride=2, padding=1)
        self.mid1 = ResBlock(base*4, base*4, time_dim)
        self.mid2 = ResBlock(base*4, base*4, time_dim)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 4, stride=2, padding=1)
        self.dec2 = ResBlock(base*4, base*2, time_dim)
        self.up1 = nn.ConvTranspose2d(base*2, base, 4, stride=2, padding=1)
        self.dec1 = ResBlock(base*2, base, time_dim)
        self.out_norm = nn.GroupNorm(8, base)
        self.out_conv = nn.Conv2d(base, 1, 3, padding=1)

    def forward(self, x, t):
        temb = self.time_mlp(time_embedding(t, self.time_dim))
        a = self.enc1(self.input(x), temb)      # B, base, 64, 64
        b = self.enc2(self.down1(a), temb)      # B, 2base, 32, 32
        c = self.down2(b)                       # B, 4base, 16, 16
        c = self.mid2(self.mid1(c, temb), temb)
        u = self.up2(c)                         # B, 2base, 32, 32
        u = self.dec2(torch.cat([u, b], dim=1), temb)
        u = self.up1(u)                         # B, base, 64, 64
        u = self.dec1(torch.cat([u, a], dim=1), temb)
        return self.out_conv(F.silu(self.out_norm(u)))

model = SmallUNet().to(device)
if gpu_count > 1:
    model = nn.DataParallel(model)

# Shape check BEFORE any training
with torch.no_grad():
    xtest = torch.randn(2, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
    ttest = torch.randint(0, TIMESTEPS, (2,), device=device)
    ytest = model(xtest, ttest)
    assert ytest.shape == xtest.shape, (xtest.shape, ytest.shape)
print(f"U-Net shape test PASSED: {tuple(ytest.shape)}")

betas = torch.linspace(1e-4, 0.02, TIMESTEPS, device=device)
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running, seen = 0.0, 0
    t0 = time.time()

    for x0 in tqdm(loader, desc=f"DDPM epoch {epoch}/{EPOCHS}"):
        x0 = x0.to(device, non_blocking=True)
        b = x0.size(0)
        t = torch.randint(0, TIMESTEPS, (b,), device=device)
        noise = torch.randn_like(x0)
        ab = alpha_bar[t].view(b, 1, 1, 1)
        xt = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type=device.type, dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            predicted_noise = model(xt, t)
            loss = F.mse_loss(predicted_noise, noise)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        running += loss.item() * b
        seen += b

    mean_loss = running / max(seen, 1)
    history.append({
        "epoch": epoch, "noise_prediction_mse": mean_loss,
        "seconds": time.time() - t0
    })
    print(f"Epoch {epoch}/{EPOCHS}: noise-prediction MSE={mean_loss:.6f}")

    core = model.module if isinstance(model, nn.DataParallel) else model
    torch.save({
        "model_state_dict": core.state_dict(),
        "epoch": epoch, "history": history,
        "image_size": IMAGE_SIZE, "timesteps": TIMESTEPS,
        "base_channels": BASE
    }, OUT / "NDCLD15_diffusion_latest.pth")

hist = pd.DataFrame(history)
hist.to_csv(OUT / "NDCLD15_diffusion_training_history.csv", index=False)

plt.figure(figsize=(7, 4))
plt.plot(hist["epoch"], hist["noise_prediction_mse"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Noise-prediction MSE")
plt.title("NDCLD15 DDPM training loss")
plt.tight_layout()
plt.savefig(OUT / "NDCLD15_diffusion_loss.png", dpi=140)
plt.close()

# Standard DDPM reverse sampling
@torch.no_grad()
def sample_ddpm(n=16):
    model.eval()
    x = torch.randn(n, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
    for ti in tqdm(range(TIMESTEPS-1, -1, -1), desc="Generating samples"):
        t = torch.full((n,), ti, dtype=torch.long, device=device)
        eps = model(x, t)
        beta = betas[ti]
        alpha = alphas[ti]
        abar = alpha_bar[ti]
        mean = (x - beta / torch.sqrt(1 - abar) * eps) / torch.sqrt(alpha)
        if ti > 0:
            x = mean + torch.sqrt(beta) * torch.randn_like(x)
        else:
            x = mean
    return x.clamp(-1, 1)

samples = sample_ddpm(16)
save_image((samples + 1) / 2, SAMPLE_DIR / "diffusion_samples_grid.png", nrow=4)
for i, x in enumerate(samples):
    save_image((x + 1) / 2, SAMPLE_DIR / f"sample_{i:02d}.png")

summary = {
    "source_images": len(images),
    "image_size": IMAGE_SIZE,
    "channels": 1,
    "epochs": EPOCHS,
    "timesteps": TIMESTEPS,
    "batch_size": BATCH_SIZE,
    "final_noise_prediction_mse": float(hist["noise_prediction_mse"].iloc[-1]),
    "device": str(device),
    "gpu_count": gpu_count,
    "shape_test": "passed",
    "conditional_generation": False,
    "verified_contact_lens_labels_available": False
}
with open(OUT / "NDCLD15_diffusion_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

(OUT / "Notebook07_Conclusion.md").write_text(
    f"""# Notebook 07 — Diffusion Model Training: Preliminary Conclusion

- Source images: {len(images):,}
- Resolution: {IMAGE_SIZE} × {IMAGE_SIZE}, grayscale
- Epochs: {EPOCHS}
- Diffusion timesteps: {TIMESTEPS}
- Final noise-prediction MSE: {summary['final_noise_prediction_mse']:.6f}
- Hardware: {device}; GPUs available: {gpu_count}
- U-Net shape test: passed

An unconditional DDPM was trained on the available image subset. Verified
contact-lens labels were unavailable, so generation was not class-conditional.
Training loss alone does not establish realism, diversity, privacy, or
downstream utility. Generated samples require separate evaluation.
"""
)

ZIP_PATH = PROJECT / "ND_Iris_Notebook07_Results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, arcname=f"notebook07_diffusion_training/{p.relative_to(OUT)}")

with zipfile.ZipFile(ZIP_PATH) as z:
    bad = z.testzip()
    archived = z.namelist()

print("\n" + "="*65)
print("NOTEBOOK 07 — DIFFUSION RESULTS")
print("="*65)
print(json.dumps(summary, indent=2))
print(f"Archived files: {len(archived)}")
print(f"ZIP size: {ZIP_PATH.stat().st_size/(1024**2):.2f} MB")
print(f"ZIP integrity: {'PASSED' if bad is None else 'FAILED: ' + str(bad)}")
print(f"Checkpoint: {OUT / 'NDCLD15_diffusion_latest.pth'}")
print(f"Conclusion report: {OUT / 'Notebook07_Conclusion.md'}")
print(f"Results ZIP: {ZIP_PATH}")
print(f"Elapsed: {(time.time()-START)/60:.2f} minutes")
assert bad is None


Device: cuda; GPUs available: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
U-Net shape test PASSED: (2, 1, 64, 64)


DDPM epoch 1/5:   0%|          | 0/97 [00:00<?, ?it/s]

Epoch 1/5: noise-prediction MSE=0.204892


DDPM epoch 2/5:   0%|          | 0/97 [00:00<?, ?it/s]

Epoch 2/5: noise-prediction MSE=0.099778


...

Epoch 4/5: noise-prediction MSE=0.080733


DDPM epoch 5/5:   0%|          | 0/97 [00:00<?, ?it/s]

Epoch 5/5: noise-prediction MSE=0.074839


Generating samples:   0%|          | 0/200 [00:00<?, ?it/s]


NOTEBOOK 07 — DIFFUSION RESULTS
{
  "source_images": 6233,
  "image_size": 64,
  "channels": 1,
  "epochs": 5,
  "timesteps": 200,
  "batch_size": 64,
  "final_noise_prediction_mse": 0.07483856994466684,
  "device": "cuda",
  "gpu_count": 2,
  "shape_test": "passed",
  "conditional_generation": false,
  "verified_contact_lens_labels_available": false
}
Archived files: 22
ZIP size: 4.60 MB
ZIP integrity: PASSED
Checkpoint: /kaggle/working/nd_iris_project/notebook07_diffusion_training/NDCLD15_diffusion_latest.pth
Conclusion report: /kaggle/working/nd_iris_project/notebook07_diffusion_training/Notebook07_Conclusion.md
Results ZIP: /kaggle/working/nd_iris_project/ND_Iris_Notebook07_Results.zip
Elapsed: 1.41 minutes


# Notebook 07 — Diffusion Model Training: Conclusion

## Objective
Train an unconditional Denoising Diffusion Probabilistic Model (DDPM) using the available NDCLD15 ocular image dataset and generate synthetic grayscale images.

## Experimental Configuration
- **Dataset:** NDCLD15
- **Source images:** 6,233
- **Training image resolution:** 64 × 64 pixels
- **Image channels:** 1 (grayscale)
- **Training epochs:** 5
- **Diffusion timesteps:** 200
- **Batch size:** 64
- **Compute:** 2 NVIDIA Tesla T4 GPUs
- **U-Net shape test:** Passed
- **Generation type:** Unconditional

## Results
The final noise-prediction mean squared error (MSE) was **0.074839**, decreasing from **0.204892** in the first epoch. This represents an approximate 63.5% reduction in training noise-prediction MSE.

The model successfully completed all five training epochs and generated synthetic samples using the 200-step reverse diffusion process. The model checkpoint, training history, loss plot, generated samples, and supporting reports were archived in `ND_Iris_Notebook07_Results.zip`. ZIP integrity verification passed.

## Limitations
The model was trained for only five epochs, so the results should be considered preliminary. A lower noise-prediction loss does not guarantee realistic, diverse, or identity-preserving synthetic images. The dataset download did not include verified contact-lens class annotations; therefore, this experiment does not demonstrate conditional contact-lens generation or verified biometric identity preservation.

## Conclusion
The DDPM training pipeline executed successfully, the U-Net passed its tensor-shape test, and synthetic images were generated and archived. These results establish a working baseline for the diffusion component of the project. The generated images require further quality and diversity evaluation before their suitability for dataset augmentation can be determined.